# LLM-Assisted Data Plotting with Gemini

This notebook teaches you how to use a Large Language Model (Gemini) to **generate plotting code from plain English descriptions**.

Instead of looking up matplotlib/seaborn/plotly syntax, you describe what you want — and the LLM writes the code for you.

### What you'll learn
- How to connect to Gemini via the Google Generative AI SDK
- How to pass data context to an LLM so it understands your dataset
- How to generate and run plotting code for **matplotlib**, **seaborn**, and **plotly**
- How to write better prompts for more precise plots

### Prerequisites
- A free Gemini API key from [Google AI Studio](https://aistudio.google.com/app/apikey)
- Store your key in Colab Secrets (the 🔑 icon in the left sidebar) under the name `GEMINI_API_KEY`

---
## Section 1 — Install & Import Dependencies

In [ ]:
# Install the Google Generative AI SDK (pre-installed in most Colab environments, but safe to re-run)
!pip install -q google-generativeai

In [ ]:
import re
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

import google.generativeai as genai
from google.colab import userdata

# Load your Gemini API key from Colab Secrets
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

# We'll use gemini-1.5-flash: fast, capable, and free-tier friendly
gemini = genai.GenerativeModel('gemini-1.5-flash')

print('Setup complete. Gemini is ready.')

---
## Section 2 — Dummy Datasets

We create three small datasets inline. In a real project you would load a CSV with `pd.read_csv()`.

| Dataset | Description | Good for |
|---|---|---|
| `df_sales` | Monthly sales by product category | bar charts, line charts |
| `df_students` | Exam scores, study hours, grade group | histograms, box plots, heatmaps |
| `df_countries` | GDP, population, life expectancy | scatter plots, bubble charts |

In [ ]:
# --- Sales data: 12 months x 4 product categories ---
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
rng = np.random.default_rng(42)

df_sales = pd.DataFrame({
    'Month':    months,
    'Electronics': rng.integers(15000, 35000, 12),
    'Clothing':    rng.integers(8000,  20000, 12),
    'Groceries':   rng.integers(20000, 40000, 12),
    'Furniture':   rng.integers(5000,  15000, 12),
})
df_sales['Total'] = df_sales[['Electronics','Clothing','Groceries','Furniture']].sum(axis=1)

print('df_sales shape:', df_sales.shape)
df_sales.head()

In [ ]:
# --- Student data: 120 students with scores, study hours, and grade group ---
n = 120
df_students = pd.DataFrame({
    'student_id':   range(1, n + 1),
    'exam_score':   np.clip(rng.normal(65, 15, n), 0, 100).round(1),
    'study_hours':  np.clip(rng.normal(6, 2.5, n), 0, 15).round(1),
    'assignments':  np.clip(rng.normal(75, 12, n), 0, 100).round(1),
    'attendance':   np.clip(rng.normal(80, 10, n), 0, 100).round(1),
    'grade_group':  rng.choice(['A','B','C','D'], n, p=[0.2, 0.35, 0.3, 0.15]),
})

print('df_students shape:', df_students.shape)
df_students.head()

In [ ]:
# --- Country data: 50 fictional countries ---
country_names = [f'Country_{i}' for i in range(1, 51)]
continents = rng.choice(['Africa','Asia','Europe','Americas','Oceania'], 50, p=[0.25,0.3,0.25,0.15,0.05])

df_countries = pd.DataFrame({
    'country':        country_names,
    'continent':      continents,
    'gdp_per_capita': np.clip(rng.lognormal(9, 1.2, 50), 500, 80000).round(),
    'life_expectancy':np.clip(rng.normal(70, 10, 50), 45, 85).round(1),
    'population_M':   np.clip(rng.lognormal(3, 1.5, 50), 0.5, 1400).round(1),
    'co2_per_capita': np.clip(rng.lognormal(1.5, 0.8, 50), 0.1, 20).round(2),
})

print('df_countries shape:', df_countries.shape)
df_countries.head()

---
## Section 3 — The Core Helper Functions

These two functions are the engine of the notebook:

1. **`build_prompt`** — Describes your DataFrame to Gemini (column names, types, a few sample rows) plus your natural-language request, and instructs the model to return *only* runnable Python code.
2. **`ask_gemini_to_plot`** — Calls the Gemini API, extracts the code block from the response, and executes it.

In [ ]:
def build_prompt(df: pd.DataFrame, df_name: str, user_request: str, library_hint: str = '') -> str:
    """Construct a prompt that gives Gemini full context about the DataFrame."""
    col_info = ', '.join([f'{col} ({dtype})' for col, dtype in zip(df.columns, df.dtypes)])
    sample_rows = df.head(3).to_string(index=False)

    library_line = f'Use the {library_hint} library.' if library_hint else ''

    prompt = textwrap.dedent(f"""
        You are a Python data visualisation expert.
        A pandas DataFrame called `{df_name}` is already defined with these columns:
          {col_info}

        First 3 rows:
        {sample_rows}

        Task: {user_request}
        {library_line}

        Rules:
        - Return ONLY a Python code block (```python ... ```).
        - Do NOT include any explanation or text outside the code block.
        - The DataFrame `{df_name}` is already loaded — do not reload or redefine it.
        - All required libraries (matplotlib, seaborn, plotly, pandas, numpy) are already imported.
        - End matplotlib/seaborn plots with `plt.tight_layout(); plt.show()`.
        - End plotly plots with `fig.show()`.
    """).strip()

    return prompt


def extract_code(text: str) -> str:
    """Pull the first Python code block out of an LLM response."""
    match = re.search(r'```(?:python)?\n(.*?)```', text, re.DOTALL)
    return match.group(1).strip() if match else text.strip()


def ask_gemini_to_plot(df: pd.DataFrame, df_name: str, user_request: str,
                       library_hint: str = '', print_code: bool = True) -> None:
    """
    Ask Gemini to write plotting code for `df`, then execute it.

    Parameters
    ----------
    df           : the DataFrame to plot
    df_name      : the variable name Gemini should use in generated code
    user_request : plain-English description of the plot you want
    library_hint : optional hint such as 'matplotlib', 'seaborn', or 'plotly'
    print_code   : if True, print the generated code before running it
    """
    prompt = build_prompt(df, df_name, user_request, library_hint)
    response = gemini.generate_content(prompt)
    code = extract_code(response.text)

    if print_code:
        print('--- Generated code ---')
        print(code)
        print('--- Running code ---')

    # Provide all common libraries in the execution namespace
    exec_globals = {
        df_name: df,
        'pd': pd, 'np': np,
        'plt': plt, 'matplotlib': matplotlib,
        'sns': sns,
        'px': px, 'go': go,
    }
    exec(code, exec_globals)  # noqa: S102

---
## Section 4 — Matplotlib Examples

Matplotlib is the most widely used Python plotting library — it gives you full control over every detail of a static figure.

In [ ]:
# --- Example 4.1: Grouped bar chart ---
# We tell Gemini which dataset, which library, and what we want.
ask_gemini_to_plot(
    df=df_sales,
    df_name='df_sales',
    user_request=(
        'Create a grouped bar chart showing monthly sales for each product category '
        '(Electronics, Clothing, Groceries, Furniture). '
        'Use a distinct color per category. Add a legend, axis labels, and a title.'
    ),
    library_hint='matplotlib',
)

In [ ]:
# --- Example 4.2: Line chart with highlighted peak ---
ask_gemini_to_plot(
    df=df_sales,
    df_name='df_sales',
    user_request=(
        'Plot the Total monthly sales as a line chart. '
        'Mark the month with the highest total sales with a red dot and annotate it '
        'with the exact value. Add axis labels and a title.'
    ),
    library_hint='matplotlib',
)

---
## Section 5 — Seaborn Examples

Seaborn is built on top of matplotlib and is designed for **statistical visualisation** — it makes it easy to show distributions, relationships, and group comparisons.

In [ ]:
# --- Example 5.1: Histogram of exam scores ---
ask_gemini_to_plot(
    df=df_students,
    df_name='df_students',
    user_request=(
        'Plot a histogram of exam_score with a KDE (kernel density estimate) overlay. '
        'Use 20 bins and a soft blue color palette. Add a vertical dashed line at the mean score. '
        'Add axis labels and a title.'
    ),
    library_hint='seaborn',
)

In [ ]:
# --- Example 5.2: Box plot by grade group ---
ask_gemini_to_plot(
    df=df_students,
    df_name='df_students',
    user_request=(
        'Create a box plot showing the distribution of exam_score for each grade_group '
        '(A, B, C, D). Order the groups from A to D on the x-axis. '
        'Color each group differently. Add axis labels and a title.'
    ),
    library_hint='seaborn',
)

In [ ]:
# --- Example 5.3: Correlation heatmap ---
ask_gemini_to_plot(
    df=df_students,
    df_name='df_students',
    user_request=(
        'Compute the correlation matrix of all numeric columns and display it as a heatmap. '
        'Annotate each cell with the correlation value rounded to 2 decimal places. '
        'Use a diverging color palette centered at 0. Add a title.'
    ),
    library_hint='seaborn',
)

---
## Section 6 — Plotly Examples

Plotly produces **interactive** charts — you can hover for exact values, zoom in, pan, and toggle series on/off. This makes it great for exploratory data analysis and web dashboards.

In [ ]:
# --- Example 6.1: Interactive bubble / scatter chart ---
ask_gemini_to_plot(
    df=df_countries,
    df_name='df_countries',
    user_request=(
        'Create an interactive scatter plot with gdp_per_capita on the x-axis (log scale) '
        'and life_expectancy on the y-axis. '
        'Size each bubble by population_M and color by continent. '
        'Add hover labels that show the country name, GDP, and life expectancy. '
        'Add axis labels and a title.'
    ),
    library_hint='plotly',
)

In [ ]:
# --- Example 6.2: Interactive stacked bar chart ---
ask_gemini_to_plot(
    df=df_sales,
    df_name='df_sales',
    user_request=(
        'Create an interactive stacked bar chart showing monthly sales broken down by '
        'Electronics, Clothing, Groceries, and Furniture. '
        'Show the Month on the x-axis. Add a legend, axis labels, and a title.'
    ),
    library_hint='plotly',
)

---
## Section 7 — Tips for Writing Better Prompts

The quality of the generated plot depends heavily on how you describe it. Here are patterns that work well:

| Vague prompt | Better prompt |
|---|---|
| "plot the sales" | "Create a bar chart of total monthly sales with months on the x-axis" |
| "show the distribution" | "Show a histogram of exam_score with 20 bins and a KDE overlay" |
| "scatter plot" | "Scatter plot: x = gdp_per_capita (log scale), y = life_expectancy, colored by continent" |

**Useful things to include in a prompt:**
- Which columns to use for x, y, color, size
- Which library to use (matplotlib / seaborn / plotly)
- Axis scaling (log, linear)
- Annotations (mean line, max point, labels)
- Color preferences ("use a red-blue diverging palette")
- Any transformations ("group by continent and take the mean")

---
## Section 8 — Your Turn! 🎨

Use the cell below to try your own prompt. Pick any of the three datasets (`df_sales`, `df_students`, `df_countries`) and describe the plot you want.

In [ ]:
# ✏️  Edit these three variables and run the cell
MY_DATASET    = df_students          # choose: df_sales, df_students, df_countries
MY_DATASET_NAME = 'df_students'      # match the variable name above as a string
MY_LIBRARY    = 'seaborn'            # choose: 'matplotlib', 'seaborn', 'plotly'
MY_REQUEST    = """
Create a scatter plot of study_hours vs exam_score, colored by grade_group.
Add a linear regression trend line for each group.
Add axis labels and a title.
"""

ask_gemini_to_plot(
    df=MY_DATASET,
    df_name=MY_DATASET_NAME,
    user_request=MY_REQUEST,
    library_hint=MY_LIBRARY,
)